In [1]:
import chromadb
from langchain_groq import ChatGroq

In [2]:
with open('api_key.txt', 'r') as key:

    my_api_key = key.read().strip()

In [3]:
llm = ChatGroq(
    temperature = 0, 
    api_key = my_api_key,
    model_name = "llama-3.3-70b-versatile"
)

In [4]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://jobs.boehringer-ingelheim.com/job/Ridgefield%2C-CT-CMC-Regulatory-Data-Science-Co-Op-%28Hybrid%29-Unit/1368819133/")


page_data = loader.load().pop().page_content

print(page_data)



USER_AGENT environment variable not set, consider setting it to identify your requests.













CMC Regulatory Data Science Co-Op (Hybrid) Job Details | BoehringerPRD
































Skip to main content
















Language 


Deutsch (Deutschland)


English (United States)


Español (España)


Français (France)


Italiano (Italia)


日本語 (日本)


Português (Brasil)


Русский язык (Россия)


简体中文 (中国大陆)





View Profile




Employee Login




























Language 


Deutsch (Deutschland)


English (United States)


Español (España)


Français (France)


Italiano (Italia)


日本語 (日本)


Português (Brasil)


Русский язык (Россия)


简体中文 (中国大陆)





View Profile




Employee Login











































                Explore our company 


Austria
Brazil
Canada (en)
Canada (fr)
China
France
Germany
Global
Japan
Mexico
Russia
South America
Spain
Taiwan
Turkey
United States of America




                Discover our careers 


Austria
Brazil
Canada (en)
Canada (fr)
China
France
Germany
Global
Japan
Mexico
Russia
South Amer

In [5]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE: 
    {page_data}
    ### INSTRUCTION : 
    The scraped text is from the careers page of a website.
    Your job is to extract the job postings and return them in JSON format containing the following 
    keys : `role`, `experience` , `skills` and `description`.
    only return the valid JSON. 
    ### VALID JSON (NO PREAMBLE) : 

"""
)



chain_extract = prompt_extract | llm 


res = chain_extract.invoke(input = {'page_data' : page_data })

print(res.content)

```json
{
  "role": "CMC Regulatory Data Science Co-Op (Hybrid)",
  "experience": "Current undergraduate, graduate or advanced degree student in good academic standing",
  "skills": [
    "Computer Science",
    "Data Management",
    "Excellent computer and information technology literacy",
    "Excellent skills in planning, organizing, and problem-solving",
    "Strong communication (verbal and written) skills",
    "Ability to work well under pressure, work in a team environment, flexibility to adapt in a changing environment",
    "Detail oriented, well organized"
  ],
  "description": "Boehringer Ingelheim is currently seeking a talented and innovative Data Science co-op to join our CMC Regulatory Affairs department located at our Ridgefield facility. As an co-op, you will utilize your data science background to optimize data platforms and develop reports for marketed products."
}
```


In [6]:
from langchain_core.output_parsers import JsonOutputParser


json_parser = JsonOutputParser()

json_res = json_parser.parse(res.content)

json_res

{'role': 'CMC Regulatory Data Science Co-Op (Hybrid)',
 'experience': 'Current undergraduate, graduate or advanced degree student in good academic standing',
 'skills': ['Computer Science',
  'Data Management',
  'Excellent computer and information technology literacy',
  'Excellent skills in planning, organizing, and problem-solving',
  'Strong communication (verbal and written) skills',
  'Ability to work well under pressure, work in a team environment, flexibility to adapt in a changing environment',
  'Detail oriented, well organized'],
 'description': 'Boehringer Ingelheim is currently seeking a talented and innovative Data Science co-op to join our CMC Regulatory Affairs department located at our Ridgefield facility. As an co-op, you will utilize your data science background to optimize data platforms and develop reports for marketed products.'}

In [7]:
import pandas as pd


df = pd.read_csv('projects.csv')
df.head()

,TechStack,Links
0,"React, Node.js",MongoDB\thttps://example.com/react-portfolio
1,"Python, Django",PostgreSQL\thttps://example.com/ecommerce-django
2,"Vue.js, Express",MySQL\thttps://example.com/vue-dashboard
3,"Angular, Firebase",TypeScript\thttps://example.com/angular-chat-app
4,"Ruby on Rails, PostgreSQL",Redis\thttps://example.com/rails-social-network


In [9]:
import uuid

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name = 'portfolio')

if not collection.count():
    for _,row in df.iterrows():
        collection.add(documents = row['TechStack'], 
                       metadatas = {
                           'Links' : row['Links']
                       },
                       ids = [str(uuid.uuid4())])

In [10]:
links = collection.query(query_texts = ['Experience in Python', 'Experience in React'], n_results = 2).get('metadatas')

links

[[{'Links': ' PostgreSQL\thttps://example.com/ecommerce-django'},
  {'Links': ' SQLite\thttps://example.com/flask-todo-api'}],
 [{'Links': ' MongoDB\thttps://example.com/react-portfolio'},
  {'Links': ' Node.js\thttps://example.com/react-native-store'}]]

In [12]:
job = json_res

job['skills']

['Computer Science',
 'Data Management',
 'Excellent computer and information technology literacy',
 'Excellent skills in planning, organizing, and problem-solving',
 'Strong communication (verbal and written) skills',
 'Ability to work well under pressure, work in a team environment, flexibility to adapt in a changing environment',
 'Detail oriented, well organized']

In [14]:
links = collection.query(query_texts = job['skills'], n_results = 2).get('metadatas')

links

[[{'Links': ' Blueprints\thttps://example.com/unreal-fps-demo'},
  {'Links': ' JavaScript\thttps://example.com/vanilla-landing-page'}],
 [{'Links': ' Node.js\thttps://example.com/graphql-music-library'},
  {'Links': ' TypeScript\thttps://example.com/angular-chat-app'}],
 [{'Links': ' WebGL\thttps://example.com/unity-browser-game'},
  {'Links': ' SQL Server\thttps://example.com/aspnet-job-board'}],
 [{'Links': ' Pandas\thttps://example.com/ml-price-predictor'},
  {'Links': ' PostgreSQL\thttps://example.com/ecommerce-django'}],
 [{'Links': ' CoreData\thttps://example.com/swift-fitness-tracker'},
  {'Links': ' JavaScript\thttps://example.com/vanilla-landing-page'}],
 [{'Links': ' SQLite\thttps://example.com/rust-url-shortener'},
  {'Links': ' WebGL\thttps://example.com/unity-browser-game'}],
 [{'Links': ' JavaScript\thttps://example.com/vanilla-landing-page'},
  {'Links': ' Blueprints\thttps://example.com/unreal-fps-demo'}]]

In [16]:
prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION 
    {job_description}

    ### INSTRUCTION: 
    You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software consulting company dedicates 
    the seamless integration of business processes through automated tools.
    Over our experience, we have empowered numerous enterprices with talented solutions, fostering scalability,
    process optimization, cost reduction, and heightened overall efficiency. 
    Your job is to write a cold email to the client regarding the job mentioned above describing the capability of Atliq
    in fulfilling their needs. 

    Also add the most relevant ones from the following the links to showcase Atliq's portfolio : {link_list}
    Remember you are Mohan, BDE at AtliQ.
    Do not provide a preamble. 
    #### EMAIL (NO PREAMBLE) : 
"""
)


chain_email = prompt_email | llm 
res = chain_email.invoke({'job_description' : str(job), 'link_list': links})

print(res.content)

Subject: Expert Data Science Solutions for CMC Regulatory Affairs

Dear Hiring Manager,

I came across the job posting for a CMC Regulatory Data Science Co-Op (Hybrid) at Boehringer Ingelheim and was impressed by the company's commitment to innovation and excellence. As a Business Development Executive at AtliQ, I believe our team can provide the necessary expertise to support your CMC Regulatory Affairs department.

At AtliQ, we specialize in leveraging AI and software solutions to optimize business processes, enhance scalability, and reduce costs. Our team of experts has a proven track record of delivering tailored solutions that meet the unique needs of our clients. In the context of data science, we have developed solutions using technologies such as Pandas (https://example.com/ml-price-predictor) and PostgreSQL (https://example.com/ecommerce-django), which can be applied to optimize data platforms and develop reports for marketed products.

Our capabilities align with the requirem